# 06 · Simulación de producción

**Patrón: batch.** El plan de refuerzo es semanal: la administración arma el horario el domingo para la semana siguiente. Las predicciones de una semana se emiten el domingo a las 23:00 (todas las features miran ≥ 7 días atrás, así que ya existen), y el target de cada día llega esa noche, después de la ingesta de las 20:30.

**Lotes:** 31 lotes diarios, del 26-10 al 25-11 (todos posteriores al entrenamiento, que termina el 25-10). Cada lote es un día de tienda (9 a 11 horas abiertas).

El flujo `kestra/produccion_lotes.yml` recorre los lotes en orden; por cada uno: (1) llegan los targets que ya existirían a la hora de emitirlo y se unen con sus predicciones → `monitoreo` de desempeño (que puede disparar `reentrenamiento`); (2) el lote se lee de `MLOPS.LOTES` y pasa por la API, que registra cada predicción en `MLOPS.PREDICCIONES`; (3) `monitoreo` de entradas.

Este notebook **no vuelve a correr la simulación** (eso es `make simulacion`): documenta lo que quedó en Snowflake, MLflow y Kestra, y demuestra el rollback, el interruptor de respaldo y la trazabilidad contra los servicios en vivo.

In [1]:
import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
import warnings; warnings.filterwarnings("ignore")
import json, time
import pandas as pd, requests
from dotenv import load_dotenv
import mlflow
from mlflow import MlflowClient

import src.pipeline as P
import src.tracking as T
from src import mlops
from src.datos import SNAPSHOT_PRODUCCION, cargar_obt

load_dotenv(RAIZ / ".env")
T.configurar()
API = "http://localhost:8010"
KESTRA = "http://localhost:8080/api/v1/main"
AUTH_KESTRA = (os.environ["KESTRA_USER"], os.environ["KESTRA_PASSWORD"])
ADMIN = {"X-Token-Admin": os.environ["TOKEN_ADMIN"]}
cliente = MlflowClient()
pd.set_option("display.width", 200, "display.max_colwidth", 90)

## 1. Los lotes y el reloj de la simulación

In [2]:
estado = mlops.leer("select * from LOTES_ESTADO order by lote_id")
lotes = mlops.leer("select lote_id, max(deriva_inducida) as deriva_inducida, count(*) as horas from LOTES group by 1")
tabla = estado.merge(lotes, on="lote_id")[["lote_id", "fecha", "horas", "emitido_en", "target_recibido_en",
                                            "n_solicitudes", "n_errores", "deriva_inducida"]]
tabla

,lote_id,fecha,horas,emitido_en,target_recibido_en,n_solicitudes,n_errores,deriva_inducida
0,1,2026-10-26,11,2026-10-25 23:00:00,2026-11-01 23:00:00,11,0,NaN
1,2,2026-10-27,11,2026-10-25 23:00:00,2026-11-01 23:00:00,11,0,NaN
2,3,2026-10-28,11,2026-10-25 23:00:00,2026-11-01 23:00:00,11,0,NaN
3,4,2026-10-29,11,2026-10-25 23:00:00,2026-11-01 23:00:00,11,0,NaN
4,5,2026-10-30,11,2026-10-25 23:00:00,2026-11-01 23:00:00,11,0,NaN
5,6,2026-10-31,11,2026-10-25 23:00:00,2026-11-01 23:00:00,11,0,NaN
6,7,2026-11-01,9,2026-10-25 23:00:00,2026-11-01 23:00:00,9,0,NaN
7,8,2026-11-02,10,2026-11-01 23:00:00,2026-11-08 23:00:00,10,0,NaN
8,9,2026-11-03,10,2026-11-01 23:00:00,2026-11-08 23:00:00,10,0,NaN
9,10,2026-11-04,11,2026-11-01 23:00:00,2026-11-08 23:00:00,11,0,NaN


In [3]:
snap = cargar_obt(ruta=SNAPSHOT_PRODUCCION)
print(f"OBT de producción: {snap.tabla_origen}, corte {snap.fecha_corte}, {len(snap.datos)} horas, hash {snap.hash[:16]}…")
assert (pd.to_datetime(estado["target_recibido_en"]) > pd.to_datetime(estado["fecha"])).all()
assert (pd.to_datetime(estado["emitido_en"]) < pd.to_datetime(estado["fecha"])).all()
print("Ningún target llegó antes de su día ni ninguna predicción se emitió después de que empezara su día.")

OBT de producción: PSET2.OBT.OBT_SALES_HOURLY, corte 2026-11-25, 1047 horas, hash 21d16405051ff355…
Ningún target llegó antes de su día ni ninguna predicción se emitió después de que empezara su día.


## 2. Deriva inducida: qué se alteró exactamente

| Lotes | Tipo | Alteración | Qué simula |
|---|---|---|---|
| 18 y 19 (12 y 13-11) | Conceptual | `sales_count` real × 2 en cada hora; **ninguna entrada cambia** | Un evento de demanda (p. ej. promoción 2×1) que el modelo no conoce: cambia la relación features → target |
| 25 (19-11) | Datos | `sales_lag_7d/14d/21d/28d`, `sales_avg_4w_same_hour` y `ventas_dia_lag_7d` reemplazados por los de la **sucursal 2** en las mismas fechas y horas; el target es el real | El join de la OBT pierde el filtro de `branch_id` en un reproceso |

La duplicación de los lotes 18-19 no se arrastra a los rezagos de los lotes siguientes: el experimento aísla su efecto en esos dos lotes.

In [4]:
real = P.agregar_features(snap.datos)
real = real[real["hour_date"] == "2026-11-19"].set_index("hour_of_day")
servido = mlops.leer("select * from LOTES where lote_id = 25 order by hour_of_day").set_index("hour_of_day")
cols = ["sales_lag_7d", "sales_avg_4w_same_hour", "ventas_dia_lag_7d"]
pd.concat({"matriz (real)": real[cols], "lote 25 (servido)": servido[cols]}, axis=1)

matriz (real)                                          lote 25 (servido)                                         
             sales_lag_7d sales_avg_4w_same_hour ventas_dia_lag_7d      sales_lag_7d sales_avg_4w_same_hour ventas_dia_lag_7d
hour_of_day                                                                                                                  
8                     4.0                   3.25              78.0               NaN                    NaN               NaN
9                     3.0                   6.00              78.0               2.0                   1.75              22.0
10                    3.0                   4.75              78.0               0.0                   1.50              22.0
11                    5.0                   7.00              78.0               3.0                   1.50              22.0
12                    6.0                   5.25              78.0               6.0                   3.50              22.0
13                   10.0                   6.25              78.0               0.0                   1.50              22.0
14                    6.0                   4.75              78.0               1.0                   0.75              22.0
15                    6.0                   6.25              78.0               3.0                   3.00              22.0
16                    9.0                   8.50              78.0               1.0                   1.75              22.0
17                   14.0                  11.00              78.0               4.0                   4.25              22.0
18                   12.0                  10.00              78.0               2.0                   1.25              22.0

In [5]:
tg = mlops.leer("select lote_id, sum(sales_count) as ventas_servidas from TARGETS where lote_id in (18, 19) group by 1")
reales = snap.datos[snap.datos["hour_date"].isin(pd.to_datetime(["2026-11-12", "2026-11-13"]))].groupby("hour_date")["sales_count"].sum()
tg.assign(ventas_reales=reales.to_numpy())

,lote_id,ventas_servidas,ventas_reales
0,18,156,78
1,19,210,105


## 3. Ejecuciones en Kestra

In [6]:
ej = requests.get(f"{KESTRA}/executions/search", params={"namespace": "proyecto_final", "size": 200}, auth=AUTH_KESTRA).json()["results"]
ej = pd.DataFrame([{"flujo": e["flowId"], "estado": e["state"]["current"], "inicio": e["state"]["startDate"][:19],
                    "duracion_s": round(pd.Timedelta(e["state"].get("duration", "PT0S")).total_seconds()), "id": e["id"]} for e in ej])
print(ej.groupby(["flujo", "estado"]).size())
ej[ej["flujo"].isin(["produccion_lotes", "reentrenamiento"]) & (ej["estado"] == "SUCCESS")].sort_values("inicio").tail(3)

flujo             estado 
ingest_erp        SUCCESS     1
monitoreo         SUCCESS    43
produccion_lotes  FAILED      1
                  SUCCESS     1
reentrenamiento   SUCCESS     1
transform         SUCCESS     1
dtype: int64


,flujo,estado,inicio,duracion_s,id
11,produccion_lotes,SUCCESS,2026-11-26T17:19:32,794,4Ka0gGp1S9MKDq3C4kFUEX
38,reentrenamiento,SUCCESS,2026-11-26T17:26:57,90,4qe96drNM8QZVMYxiQOltH


## 4. Predicciones registradas y versión que las produjo

In [7]:
pred = mlops.leer("select * from PREDICCIONES where lote_id is not null")
print(f"{len(pred)} predicciones de la simulación")
display(pred.groupby(["modelo_version", "modelo_run_id"]).agg(lotes=("lote_id", lambda s: f"{s.min()}-{s.max()}"), predicciones=("id_prediccion", "count"),
                                                         respaldo=("respaldo", "sum"), latencia_p95_ms=("latencia_ms", lambda s: s.quantile(.95))).round(1))
pred[["id_prediccion", "ts", "lote_id", "hour_date", "hour_of_day", "ventas_previstas", "decision", "modelo_version", "latencia_ms"]].head()

331 predicciones de la simulación


,,lotes,predicciones,respaldo,latencia_p95_ms
modelo_version,modelo_run_id,,,,
1,b2a93515cf00415f9b32336eadbe00bf,1-21,223,0,31.6
3,2bb5e21c55de4418bfe03dcf600dea6c,22-31,108,0,24.2


,id_prediccion,ts,lote_id,hour_date,hour_of_day,ventas_previstas,decision,modelo_version,latencia_ms
0,49a3d3c6-29f6-4847-9867-d6b4c9882293,2026-11-26 17:19:52.182118+00:00,1,2026-10-26,8,0.638799,no_reforzar,1,43.968590
1,f71674c8-b127-4580-8591-4a57f4f47720,2026-11-26 17:19:52.196877+00:00,1,2026-10-26,9,2.542906,no_reforzar,1,10.549428
2,d89bb6b5-02d2-454e-916a-89bb38f59b41,2026-11-26 17:19:52.205250+00:00,1,2026-10-26,10,3.472838,no_reforzar,1,6.432965
3,e453b11a-cdb5-4f3d-90dd-eac239045008,2026-11-26 17:19:52.212743+00:00,1,2026-10-26,11,5.012765,no_reforzar,1,5.801632
4,c7b8e28f-658d-44c0-a74d-0a148b198330,2026-11-26 17:19:52.219761+00:00,1,2026-10-26,12,4.444345,no_reforzar,1,5.544715


## 5. Reentrenamiento: disparo, comparación y decisión

In [8]:
reent = mlops.leer("select * from REENTRENAMIENTOS order by creado_en")
reent[["lote_disparo", "motivo", "version_champion", "version_challenger", "lote_evaluacion", "costo_champion", "costo_challenger",
       "sigma_costo", "wape_champion", "wape_challenger", "sigma_wape", "recomendacion", "decision", "motivo_decision"]].T

,0
lote_disparo,21
motivo,"desempeño: WAPE sobre el umbral en los lotes seguidos [18, 19]"
version_champion,1
version_challenger,3
lote_evaluacion,21
costo_champion,31.41
costo_challenger,15.14
sigma_costo,7.227743
wape_champion,0.614205
wape_challenger,0.651929


In [9]:
exp = mlflow.get_experiment_by_name(T.EXPERIMENTO)
r = mlflow.search_runs([exp.experiment_id], filter_string="tags.fase = 'reentrenamiento'")
cols = ["run_id", "tags.motivo", "params.ventana", "params.ventana_inicio", "params.ventana_fin", "params.lote_evaluacion",
        "params.lotes_excluidos_alerta_datos", "params.lotes_excluidos_decision", "params.alpha", "params.umbral_decision",
        "metrics.cv_wape_mean", "metrics.cv_wape_std", "tags.recomendacion", "tags.decision"]
r[cols].T

,0
run_id,2bb5e21c55de4418bfe03dcf600dea6c
tags.motivo,"desempeño: WAPE sobre el umbral en los lotes seguidos [18, 19]"
params.ventana,todo el histórico etiquetado
params.ventana_inicio,2026-08-20
params.ventana_fin,2026-11-14
params.lote_evaluacion,21
params.lotes_excluidos_alerta_datos,-
params.lotes_excluidos_decision,-
params.alpha,1.0
params.umbral_decision,5.25


In [10]:
for alias in ["champion", "anterior"]:
    v = cliente.get_model_version_by_alias(T.MODELO_REGISTRADO, alias)
    print(f"@{alias:<9} -> v{v.version}  run {v.run_id}  umbral {v.tags.get('umbral_decision')}")

@champion  -> v3  run 2bb5e21c55de4418bfe03dcf600dea6c  umbral 5.25
@anterior  -> v1  run b2a93515cf00415f9b32336eadbe00bf  umbral 6.75


## 6. Rollback sin reconstruir la imagen

Se mueve el alias `champion` a la versión anterior (v1) y se espera a que la API lo tome (revisa el alias cada 30 s); luego se vuelve a v3. En ningún momento se toca la imagen ni el contenedor.

In [11]:
def esperar_version(objetivo: int, limite_s: int = 90) -> dict:
    inicio = time.time()
    while time.time() - inicio < limite_s:
        info = requests.get(f"{API}/modelo").json()
        if info["version"] == objetivo:
            return {**info, "segundos": round(time.time() - inicio, 1)}
        time.sleep(3)
    raise TimeoutError(f"la API no tomó v{objetivo} en {limite_s} s")

actual = cliente.get_model_version_by_alias(T.MODELO_REGISTRADO, "champion").version
anterior = cliente.get_model_version_by_alias(T.MODELO_REGISTRADO, "anterior").version
imagen = os.popen("docker inspect proyecto_final-api-1 --format '{{.Image}} {{.State.StartedAt}}'").read().strip()
print("Antes:   /modelo ->", {k: requests.get(f"{API}/modelo").json()[k] for k in ["version", "run_id"]})

cliente.set_registered_model_alias(T.MODELO_REGISTRADO, "champion", anterior)        # rollback
info = esperar_version(int(anterior))
print(f"Rollback: /modelo -> v{info['version']} ({info['run_id']}) en {info['segundos']} s")

cliente.set_registered_model_alias(T.MODELO_REGISTRADO, "champion", actual)          # se restaura
info = esperar_version(int(actual))
print(f"Restaurado: /modelo -> v{info['version']} ({info['run_id']}) en {info['segundos']} s")
assert imagen == os.popen("docker inspect proyecto_final-api-1 --format '{{.Image}} {{.State.StartedAt}}'").read().strip()
print("Misma imagen y mismo contenedor (no se reconstruyó ni se reinició):", imagen[:30], "…")

Antes:   /modelo -> {'version': 3, 'run_id': '2bb5e21c55de4418bfe03dcf600dea6c'}


Rollback: /modelo -> v1 (b2a93515cf00415f9b32336eadbe00bf) en 24.2 s


Restaurado: /modelo -> v3 (2bb5e21c55de4418bfe03dcf600dea6c) en 30.2 s
Misma imagen y mismo contenedor (no se reconstruyó ni se reinició): sha256:9ec2b2c757fedb031457f4b …


## 7. Contrato e interruptor de respaldo, en vivo

In [12]:
ejemplo = json.loads((RAIZ / "tests/fixtures/ejemplo.json").read_text())
ok = requests.post(f"{API}/predecir", json=ejemplo)
print(ok.status_code, ok.json())
malo = requests.post(f"{API}/predecir", json={**ejemplo, "hour_of_day": 22, "staff_present": "dos"})
print(malo.status_code, [(e["loc"][-1], e["msg"]) for e in malo.json()["detail"]])

200

 {'id_prediccion': '1ba82386-eeeb-45b4-9070-f5a07c826d49', 'ventas_previstas': 7.6481, 'carga_prevista': 3.824, 'umbral_carga': 5.25, 'decision': 'no_reforzar', 'accion': 'Sin refuerzo el 2026-10-28 de 16:00-17:00: alcanza el personal programado.', 'version_modelo': 3, 'alias': 'champion', 'run_id': '2bb5e21c55de4418bfe03dcf600dea6c', 'respaldo': False, 'motivo_respaldo': None}
422 [('hour_of_day', 'Input should be 8, 9, 10, 11, 12, 13, 14, 15, 16, 17 or 18'), ('staff_present', 'Input should be a valid integer')]


In [13]:
print(requests.post(f"{API}/respaldo", json={"activo": True, "motivo": "demostración del interruptor"}).status_code, "<- sin token")
print(requests.post(f"{API}/respaldo", json={"activo": True, "motivo": "demostración del interruptor"}, headers=ADMIN).json())
r = requests.post(f"{API}/predecir", json=ejemplo).json()
print({k: r[k] for k in ["ventas_previstas", "umbral_carga", "decision", "version_modelo", "respaldo", "motivo_respaldo"]})
assert r["respaldo"] and r["ventas_previstas"] == ejemplo["sales_avg_4w_same_hour"]
print(requests.post(f"{API}/respaldo", json={"activo": False, "motivo": "fin de la demostración"}, headers=ADMIN).json())
r = requests.post(f"{API}/predecir", json=ejemplo).json()
print({k: r[k] for k in ["ventas_previstas", "decision", "version_modelo", "respaldo"]})

401 <- sin token
{'estado': 'degradado', 'modelo_cargado': True, 'modo_respaldo': True, 'detalle': 'respaldo activo: demostración del interruptor'}
{'ventas_previstas': 4.0, 'umbral_carga': 8.0, 'decision': 'no_reforzar', 'version_modelo': None, 'respaldo': True, 'motivo_respaldo': 'interruptor de respaldo activo (demostración del interruptor)'}
{'estado': 'ok', 'modelo_cargado': True, 'modo_respaldo': False, 'detalle': None}
{'ventas_previstas': 7.6481, 'decision': 'no_reforzar', 'version_modelo': 3, 'respaldo': False}


## 8. Trazabilidad: de una predicción a las tablas Gold

Una fila de `PREDICCIONES` → versión del modelo y `run_id` → corrida de MLflow (datos con que se entrenó: hash del snapshot de la OBT y lotes de producción) → la OBT (`PSET2.OBT.OBT_SALES_HOURLY`, construida por Spark) → las tablas Gold de las que se deriva.

In [14]:
fila = pred[pred["modelo_version"] == pred["modelo_version"].max()].sort_values("ts").iloc[0]
run = cliente.get_run(fila["modelo_run_id"])
p = run.data.params
print(f"Predicción {fila['id_prediccion']} · lote {fila['lote_id']} · {fila['hour_date']} {fila['hour_of_day']}:00 · {fila['ventas_previstas']:.2f} ventas · {fila['decision']}")
print(f"  └─ modelo {fila['modelo_nombre']} v{int(fila['modelo_version'])} · run_id {fila['modelo_run_id']} · fase {run.data.tags['fase']} · commit {run.data.tags['git_commit'][:10]}")
print(f"      ├─ datos de entrenamiento: {p['datos_tabla_origen']} · {p['datos_filas_train']} filas · ventana {p['ventana_inicio']} a {p['ventana_fin']} · hash {p['datos_hash'][:16]}…")
print(f"      ├─ snapshot de la OBT: hash {p['datos_snapshot_hash'][:16]}… (data/obt_matriz.parquet, corte 2026-11-07) + lotes {p['lotes_produccion_usados']}")
print(f"      ├─ artefacto con las filas exactas: runs:/{run.info.run_id}/datos/entrenamiento.parquet")
print(f"      └─ champion anterior: v{p['version_champion']} · run_id {p['run_id_champion']}")
padre = cliente.get_run(p["run_id_champion"]).data.params
print(f"          └─ entrenado con {padre['datos_tabla_origen']} @ {padre['datos_fecha_corte']} · hash {padre['datos_hash'][:16]}…")
print("              └─ OBT construida por spark/build_obt.py desde GOLD.FCT_STORE_HOURS, GOLD.FCT_SALES_LINES,")
print("                 GOLD.DIM_DATE, GOLD.DIM_BRANCH y GOLD.DIM_HOUR (dbt: Bronze -> Silver -> Gold)")

Predicción bb4235c8-7dd2-4033-a3da-885f575b2bec · lote 22 · 2026-11-16 8:00 · 0.81 ventas · no_reforzar
  └─ modelo ventas-hora-matriz-modelo v3 · run_id 2bb5e21c55de4418bfe03dcf600dea6c · fase reentrenamiento · commit 84fa222f3f
      ├─ datos de entrenamiento: PSET2.OBT.OBT_SALES_HOURLY + PSET2.MLOPS.LOTES/TARGETS · 930 filas · ventana 2026-08-20 a 2026-11-14 · hash 255b5a5e43e3913e…
      ├─ snapshot de la OBT: hash d40705c0073c3866… (data/obt_matriz.parquet, corte 2026-11-07) + lotes 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
      ├─ artefacto con las filas exactas: runs:/2bb5e21c55de4418bfe03dcf600dea6c/datos/entrenamiento.parquet
      └─ champion anterior: v1 · run_id b2a93515cf00415f9b32336eadbe00bf
          └─ entrenado con PSET2.OBT.OBT_SALES_HOURLY @ 2026-11-07 · hash d40705c0073c3866…
              └─ OBT construida por spark/build_obt.py desde GOLD.FCT_STORE_HOURS, GOLD.FCT_SALES_LINES,
                 GOLD.DIM_DATE, GOLD.DIM_BRANCH y GOLD.DIM_HOUR (dbt: Bronze 